# 11.1 Challenge problems across Core AI: practice notebook

Twenty one hard problems in the style of top university exams, one set for each step of Core AI, then the proofs examiners ask for and a guide for choosing a method.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### A* with an inconsistent heuristic

In [ ]:
# A* graph search with an admissible but inconsistent heuristic.
# Without reopening closed nodes it returns a path that is not the cheapest.
import heapq
edges = {'S': [('A', 4), ('B', 1)], 'B': [('A', 1)], 'A': [('G', 4)], 'G': []}
h = {'S': 0, 'A': 0, 'B': 5, 'G': 0}       # h(B) = 5 > cost(B, A) + h(A) = 1

def astar(reopen):
    frontier = [(h['S'], 0, 'S', ['S'])]
    best_g = {}
    while frontier:
        f, g, node, path = heapq.heappop(frontier)
        if node == 'G':
            return g, path
        if node in best_g and (not reopen or best_g[node] <= g):
            continue                       # already expanded
        best_g[node] = g
        for nxt, cost in edges[node]:
            heapq.heappush(frontier, (g + cost + h[nxt], g + cost, nxt, path + [nxt]))

print("no reopening:  ", astar(False))
print("with reopening:", astar(True))

### Alpha beta with a leaf counter

In [ ]:
# Alpha beta on the three by three tree from AIMA figure 5.5. Counts the leaves it looks at.
leaves_seen = []

def alphabeta(node, alpha, beta, is_max):
    if type(node) == int:
        leaves_seen.append(node)
        return node
    if is_max:
        value = -1000
        for child in node:
            value = max(value, alphabeta(child, alpha, beta, False))
            if value >= beta:
                return value               # MIN above will never allow this
            alpha = max(alpha, value)
    else:
        value = 1000
        for child in node:
            value = min(value, alphabeta(child, alpha, beta, True))
            if value <= alpha:
                return value               # MAX above already has better
            beta = min(beta, value)
    return value

tree = [[3, 12, 8], [2, 4, 6], [14, 5, 2]]
print("value:", alphabeta(tree, -1000, 1000, True))
print("leaves seen:", leaves_seen, "of 9")

### AC-3 on X < Y < Z

In [ ]:
# AC-3 on X < Y < Z with every domain {1, 2, 3}.
domains = {'X': [1, 2, 3], 'Y': [1, 2, 3], 'Z': [1, 2, 3]}
less = [('X', 'Y'), ('Y', 'Z')]                 # first < second
arcs = []
for a, b in less:
    arcs.append((a, b))
    arcs.append((b, a))

def ok(a, va, b, vb):
    if (a, b) in less:
        return va < vb
    return vb < va

queue = list(arcs)
while queue:
    a, b = queue.pop(0)
    keep = [va for va in domains[a] if any(ok(a, va, b, vb) for vb in domains[b])]
    if len(keep) < len(domains[a]):
        print("revise", a, "against", b, ":", domains[a], "becomes", keep)
        domains[a] = keep
        for c, d in arcs:
            if d == a and c != b:
                queue.append((c, d))
print(domains)

### Value iteration as a contraction

In [ ]:
# Value iteration is a contraction: the biggest change shrinks by a factor gamma each sweep.
# Two states. In s0, "stay" gives 1 and stays, "go" gives 0 and moves to s1. In s1 every action gives 2 and stays.
gamma = 0.9
V = [0.0, 0.0]
for k in range(1, 8):
    new0 = max(1 + gamma * V[0], 0 + gamma * V[1])
    new1 = 2 + gamma * V[1]
    change = max(abs(new0 - V[0]), abs(new1 - V[1]))
    V = [new0, new1]
    print("sweep", k, "V =", [round(v, 3) for v in V], "change =", round(change, 3))
print("exact: V(s1) =", round(2 / (1 - gamma), 3), " V(s0) =", round(max(1 / (1 - gamma), gamma * 2 / (1 - gamma)), 3))

## Your turn

### Task 1: Count the nodes of BFS and IDS

Write `bfs_nodes(b, d)` that returns b + b² + … + bᵈ, and `ids_nodes(b, d)` that returns the sum of (d + 1 − i) bⁱ for i from 1 to d.

Hint: Loop over the depth i from 1 to d and add up the terms.

In [ ]:
def bfs_nodes(b, d):
    # your code here
    pass

def ids_nodes(b, d):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('bfs_nodes(10, 5)', 111110), ('ids_nodes(10, 5)', 123450), ('ids_nodes(2, 3)', 22), ('bfs_nodes(3, 1)', 3)])

### Task 2: Check a heuristic

Write `bad_edges(edges, h, goal)`. `edges` is a list of (from, to, cost) and `h` a dict. Return the list of edges (from, to) where h(from) &gt; cost + h(to), in the order given, followed by True if h(goal) == 0 else False.

Hint: Test each edge with the consistency rule, then add the goal test at the end.

In [ ]:
def bad_edges(edges, h, goal):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("bad_edges([('S', 'A', 4), ('S', 'B', 1), ('B', 'A', 1), ('A', 'G', 4)], {'S': 0, 'A': 0, 'B': 5, 'G': 0}, 'G')", [['B', 'A'], True]), ("bad_edges([('S', 'G', 2)], {'S': 2, 'G': 0}, 'G')", [True]), ("bad_edges([('S', 'G', 2)], {'S': 4, 'G': 1}, 'G')", [['S', 'G'], False])])

### Task 3: Alpha beta with a leaf counter

Write `alphabeta(node, alpha, beta, is_max, seen)` for a tree of nested lists with numbers at the leaves. Append every leaf you read to the list `seen`, prune with α and β, and return the value.

Hint: Copy minimax, then stop the loop at a MAX node when value ≥ β, and at a MIN node when value ≤ α.

In [ ]:
def alphabeta(node, alpha, beta, is_max, seen):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('alphabeta([[3, 12, 8], [2, 4, 6], [14, 5, 2]], -1000, 1000, True, [])', 3), ('(lambda s: (alphabeta([[3, 12, 8], [2, 4, 6], [14, 5, 2]], -1000, 1000, True, s), len(s)))([])', [3, 7]), ('(lambda s: (alphabeta([[2, 3], [5, 9], [0, 1], [7, 5]], -1000, 1000, False, s), s))([])', [1, [2, 3, 5, 0, 1, 7]]), ('alphabeta([[[2, 3], [5, 9]], [[0, 1], [7, 5]]], -1000, 1000, True, [])', 3)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Count the nodes of BFS and IDS

Test 3: 3 × 2 + 2 × 4 + 1 × 8 = 22.

In [ ]:
def bfs_nodes(b, d):
    total = 0
    for i in range(1, d + 1):
        total = total + b ** i
    return total

def ids_nodes(b, d):
    total = 0
    for i in range(1, d + 1):
        total = total + (d + 1 - i) * b ** i
    return total

In [ ]:
check([('bfs_nodes(10, 5)', 111110), ('ids_nodes(10, 5)', 123450), ('ids_nodes(2, 3)', 22), ('bfs_nodes(3, 1)', 3)])

### Solution to task 2: Check a heuristic

An empty list before True means h is consistent.

In [ ]:
def bad_edges(edges, h, goal):
    out = []
    for a, b, cost in edges:
        if h[a] > cost + h[b]:
            out.append((a, b))
    out.append(h[goal] == 0)
    return out

In [ ]:
check([("bad_edges([('S', 'A', 4), ('S', 'B', 1), ('B', 'A', 1), ('A', 'G', 4)], {'S': 0, 'A': 0, 'B': 5, 'G': 0}, 'G')", [['B', 'A'], True]), ("bad_edges([('S', 'G', 2)], {'S': 2, 'G': 0}, 'G')", [True]), ("bad_edges([('S', 'G', 2)], {'S': 4, 'G': 1}, 'G')", [['S', 'G'], False])])

### Solution to task 3: Alpha beta with a leaf counter

Test 4 is the alpha beta example of the slides, with root value 3.

In [ ]:
def alphabeta(node, alpha, beta, is_max, seen):
    if type(node) == int:
        seen.append(node)
        return node
    if is_max:
        value = -1000
        for child in node:
            value = max(value, alphabeta(child, alpha, beta, False, seen))
            if value >= beta:
                return value
            alpha = max(alpha, value)
        return value
    value = 1000
    for child in node:
        value = min(value, alphabeta(child, alpha, beta, True, seen))
        if value <= alpha:
            return value
        beta = min(beta, value)
    return value

In [ ]:
check([('alphabeta([[3, 12, 8], [2, 4, 6], [14, 5, 2]], -1000, 1000, True, [])', 3), ('(lambda s: (alphabeta([[3, 12, 8], [2, 4, 6], [14, 5, 2]], -1000, 1000, True, s), len(s)))([])', [3, 7]), ('(lambda s: (alphabeta([[2, 3], [5, 9], [0, 1], [7, 5]], -1000, 1000, False, s), s))([])', [1, [2, 3, 5, 0, 1, 7]]), ('alphabeta([[[2, 3], [5, 9]], [[0, 1], [7, 5]]], -1000, 1000, True, [])', 3)])